In [ ]:
#!/usr/bin/env python3
# Simple Bitcoin Keyspace Analyzer - No Batches
# Just keeps all 999 addresses in memory and tracks new bests

# Install required packages
!pip install -q base58 colorama tqdm ecdsa pycryptodome

import hashlib
import base58
import os
import time
from colorama import Fore, init
from tqdm import tqdm
from google.colab import files
import binascii
import ecdsa
from Crypto.Hash import RIPEMD160

# Initialize colorama
init(autoreset=True)

# Configuration
RANGE_START = 40_000_000  # Start of key range to analyze
RANGE_END = 71_000_000    # End of key range to analyze
STEP_SIZE = 100_000       # Progress updates every this many keys
OUTPUT_DIR = "keyspace_analysis"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# File paths
LOG_FILE = f"{OUTPUT_DIR}/analysis_log.txt"
NEW_BESTS_FILE = f"{OUTPUT_DIR}/new_bests.csv"
FINAL_REPORT = f"{OUTPUT_DIR}/final_report.txt"

# Function to generate Bitcoin address from private key integer
def generate_address(private_key_int):
    try:
        # Format private key as hex
        private_key_hex = format(private_key_int, '064x')

        # Convert to ECDSA key
        sk = ecdsa.SigningKey.from_string(bytes.fromhex(private_key_hex), curve=ecdsa.SECP256k1)
        vk = sk.get_verifying_key()

        # Create public key and hash
        public_key = b'\x04' + vk.to_string()
        sha256_hash = hashlib.sha256(public_key).digest()

        # Apply RIPEMD160
        ripemd160_hasher = RIPEMD160.new()
        ripemd160_hasher.update(sha256_hash)
        hash160 = ripemd160_hasher.digest()

        # Create address with version and checksum
        versioned_hash = b'\x00' + hash160
        checksum = hashlib.sha256(hashlib.sha256(versioned_hash).digest()).digest()[:4]
        address_bytes = versioned_hash + checksum

        # Encode as Base58
        address = base58.b58encode(address_bytes).decode('utf-8')
        return address, hash160
    except Exception as e:
        print(f"Error generating address for key {private_key_int}: {str(e)}")
        return None, None

# Function to calculate Hamming distance between two binary objects
def hamming_distance(hash1, hash2):
    try:
        # Ensure both hashes are of same length
        if not hash1 or not hash2 or len(hash1) != len(hash2):
            return 160  # Maximum possible Hamming distance for 160-bit hash

        # Calculate Hamming distance bit by bit
        hamming = 0
        for a, b in zip(hash1, hash2):
            xor_byte = a ^ b
            # Count bits in XOR result
            while xor_byte:
                hamming += xor_byte & 1
                xor_byte >>= 1
        return hamming
    except Exception as e:
        print(f"Error calculating Hamming distance: {str(e)}")
        return 160

# Simple function to load Bitcoin addresses
def load_addresses():
    print(f"{Fore.CYAN}Loading target Bitcoin addresses...")

    try:
        # Check if addresses.txt exists
        if os.path.exists("addresses.txt"):
            with open("addresses.txt", 'r') as f:
                addresses = [line.strip() for line in f if line.strip()]
        else:
            # Prompt for upload
            print(f"{Fore.CYAN}Upload a text file with Bitcoin addresses (one per line)")
            uploaded = files.upload()

            if not uploaded:
                print(f"{Fore.RED}No file uploaded. Exiting.")
                return [], {}

            filename = list(uploaded.keys())[0]
            with open(filename, 'r') as f:
                addresses = [line.strip() for line in f if line.strip()]

        # Validate addresses and extract hash160 values
        valid_addresses = []
        hash160s = {}

        for addr in addresses:
            if not addr.startswith('1'):  # Only P2PKH addresses
                continue

            try:
                decoded = base58.b58decode(addr)
                if len(decoded) != 25:
                    continue

                hash160 = decoded[1:-4]  # Extract RIPEMD160 hash
                valid_addresses.append(addr)
                hash160s[addr] = hash160
            except Exception as e:
                print(f"Error decoding address {addr}: {str(e)}")
                continue

        if not valid_addresses:
            print(f"{Fore.RED}No valid P2PKH addresses found.")
            return [], {}

        print(f"{Fore.GREEN}Loaded {len(valid_addresses)} valid addresses for comparison.")
        return valid_addresses, hash160s

    except Exception as e:
        print(f"{Fore.RED}Error loading addresses: {str(e)}")
        return [], {}

# Function to initialize result tracking
def initialize_tracking(addresses):
    # Initialize tracking dictionaries - all start with max distance
    best_distances = {addr: 160 for addr in addresses}
    best_keys = {addr: None for addr in addresses}
    best_addresses = {addr: None for addr in addresses}

    # Initialize log file
    with open(LOG_FILE, 'w') as f:
        f.write("=== BITCOIN KEYSPACE ANALYSIS - SIMPLE VERSION ===\n")
        f.write(f"Started: {time.ctime()}\n")
        f.write(f"Analyzing key range: {RANGE_START:,} to {RANGE_END:,}\n")
        f.write(f"Target addresses: {len(addresses)}\n\n")
        f.write("Target addresses:\n")
        for addr in addresses:
            f.write(f"  {addr}\n")
        f.write("\n")

    # Initialize new bests file
    with open(NEW_BESTS_FILE, 'w') as f:
        f.write("Timestamp,TargetAddress,PrivateKey,GeneratedAddress,HammingDistance,PreviousBest,Improvement,Range\n")

    return best_distances, best_keys, best_addresses

# Main analysis function
def analyze_range(start_key, end_key, target_addresses, target_hash160s):
    # Initialize tracking
    best_distances, best_keys, best_addresses = initialize_tracking(target_addresses)

    # Stats tracking
    improvements_count = 0
    start_time = time.time()

    # Create progress bar
    total_keys = end_key - start_key
    progress_bar = tqdm(total=total_keys, desc="Analyzing keys")

    # Process each key in range
    current_key = start_key
    last_update = start_time

    while current_key < end_key:
        # Generate address from private key
        address, hash160 = generate_address(current_key)

        if address and hash160:
            # Compare against all target addresses
            for target_addr, target_hash160 in target_hash160s.items():
                # Calculate Hamming distance
                dist = hamming_distance(hash160, target_hash160)

                # Check if this is a new best
                if dist < best_distances[target_addr]:
                    prev_dist = best_distances[target_addr]
                    improvement = prev_dist - dist

                    # Update best results
                    best_distances[target_addr] = dist
                    best_keys[target_addr] = current_key
                    best_addresses[target_addr] = address

                    # Get current range in millions
                    current_range = f"{current_key // 1_000_000}M"

                    # Print new best immediately
                    print(f"\n{Fore.GREEN}🔥 NEW BEST! Target: {target_addr}")
                    print(f"{Fore.GREEN}    Previous best: {prev_dist} bits")
                    print(f"{Fore.GREEN}    New best: {dist} bits (improved by {improvement} bits)")
                    print(f"{Fore.GREEN}    Key: {current_key} (hex: {format(current_key, '064x')})")
                    print(f"{Fore.GREEN}    Generated address: {address}")
                    print(f"{Fore.GREEN}    Range: {current_range}")

                    # Log the new best
                    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")

                    with open(NEW_BESTS_FILE, 'a') as f:
                        f.write(f"{timestamp},{target_addr},{format(current_key, '064x')}," +
                                f"{address},{dist},{prev_dist},{improvement},{current_range}\n")

                    with open(LOG_FILE, 'a') as f:
                        f.write(f"[{timestamp}] NEW BEST for {target_addr}\n")
                        f.write(f"  Previous best: {prev_dist} bits\n")
                        f.write(f"  New best: {dist} bits (improved by {improvement} bits)\n")
                        f.write(f"  Key: {current_key} (hex: {format(current_key, '064x')})\n")
                        f.write(f"  Generated address: {address}\n")
                        f.write(f"  Range: {current_range}\n\n")

                    improvements_count += 1

        # Update progress every STEP_SIZE keys
        if current_key % STEP_SIZE == 0:
            # Update progress bar
            progress_bar.update(STEP_SIZE)

            # Print stats every minute
            current_time = time.time()
            if current_time - last_update >= 60:
                keys_processed = current_key - start_key
                elapsed = current_time - start_time
                keys_per_second = keys_processed / elapsed if elapsed > 0 else 0
                under_40_count = sum(1 for dist in best_distances.values() if dist < 40)

                range_str = f"{(current_key // 1_000_000)}M"
                print(f"\n{Fore.CYAN}Progress Update - Range: {range_str}")
                print(f"{Fore.CYAN}  Keys processed: {keys_processed:,} ({keys_per_second:.2f} keys/sec)")
                print(f"{Fore.CYAN}  Improvements found: {improvements_count}")
                print(f"{Fore.CYAN}  Addresses with Hamming < 40: {under_40_count}/{len(target_addresses)}")

                # Estimate remaining time
                remaining_keys = end_key - current_key
                est_remaining_time = remaining_keys / keys_per_second if keys_per_second > 0 else 0
                print(f"{Fore.CYAN}  Estimated remaining time: {est_remaining_time:.2f} seconds")

                last_update = current_time

        # Move to next key
        current_key += 1

    # Close progress bar
    progress_bar.close()

    # Generate final report
    generate_report(target_addresses, best_distances, best_keys, best_addresses)

    # Show final stats
    elapsed = time.time() - start_time
    print(f"\n{Fore.GREEN}Analysis completed in {elapsed:.2f} seconds")
    print(f"{Fore.GREEN}Found {improvements_count} improvements")
    under_40_count = sum(1 for dist in best_distances.values() if dist < 40)
    print(f"{Fore.GREEN}Addresses with Hamming < 40: {under_40_count}/{len(target_addresses)}")

    # Return results
    return best_distances, best_keys, best_addresses

# Function to generate final report
def generate_report(target_addresses, best_distances, best_keys, best_addresses):
    try:
        with open(FINAL_REPORT, 'w') as f:
            f.write("=== BITCOIN KEYSPACE ANALYSIS - FINAL REPORT ===\n")
            f.write(f"Generated: {time.ctime()}\n")
            f.write(f"Key range analyzed: {RANGE_START:,} to {RANGE_END:,}\n")
            f.write(f"Target addresses: {len(target_addresses)}\n\n")

            # Overall statistics
            min_hamming = min(best_distances.values()) if best_distances else 160
            under_40_count = sum(1 for dist in best_distances.values() if dist < 40)

            f.write(f"Overall minimum Hamming distance: {min_hamming} bits\n")
            f.write(f"Addresses with Hamming distance < 40: {under_40_count}/{len(target_addresses)}\n\n")

            # Sort addresses by Hamming distance
            sorted_addresses = sorted(
                [(addr, best_distances[addr]) for addr in target_addresses],
                key=lambda x: x[1]
            )

            # Best results section
            f.write("Best results:\n\n")

            for addr, dist in sorted_addresses:
                if dist < 160:  # Only include addresses with actual matches
                    f.write(f"Target address: {addr}\n")
                    f.write(f"Best Hamming distance: {dist} bits\n")
                    if best_keys[addr] is not None:
                        f.write(f"Private key: {best_keys[addr]} (hex: {format(best_keys[addr], '064x')})\n")
                        f.write(f"Generated address: {best_addresses[addr]}\n")
                    f.write("\n")

            # Summary by Hamming distance ranges
            f.write("Summary by Hamming distance ranges:\n\n")
            ranges = [(0, 30), (30, 35), (35, 40), (40, 45), (45, 50), (50, 160)]

            for low, high in ranges:
                count = sum(1 for dist in best_distances.values() if low <= dist < high)
                f.write(f"  {low}-{high} bits: {count} addresses\n")

        print(f"{Fore.GREEN}Final report saved to {FINAL_REPORT}")

    except Exception as e:
        print(f"{Fore.RED}Error generating report: {str(e)}")

# Main function
def main():
    print(f"""
{Fore.CYAN}╔══════════════════════════════════════════════════════════════════╗
{Fore.CYAN}║                                                                  ║
{Fore.CYAN}║  {Fore.GREEN}SIMPLE BITCOIN KEYSPACE ANALYZER{Fore.CYAN}                            ║
{Fore.CYAN}║  {Fore.YELLOW}Analyzing Range 40M-71M (No Batching){Fore.CYAN}                      ║
{Fore.CYAN}╚══════════════════════════════════════════════════════════════════╝
    """)

    # Load addresses
    target_addresses, target_hash160s = load_addresses()

    if not target_addresses:
        print(f"{Fore.RED}No valid addresses loaded. Exiting.")
        return

    print(f"{Fore.GREEN}Starting simple keyspace analysis - showing only new best matches")
    print(f"{Fore.GREEN}Range: {RANGE_START:,} to {RANGE_END:,}")
    print(f"{Fore.GREEN}Tracking {len(target_addresses)} addresses in memory")

    # Run analysis
    best_distances, best_keys, best_addresses = analyze_range(
        RANGE_START, RANGE_END, target_addresses, target_hash160s
    )

    # Provide download links
    print(f"\n{Fore.CYAN}Download results:")
    try:
        files.download(FINAL_REPORT)
        files.download(NEW_BESTS_FILE)
        files.download(LOG_FILE)
    except Exception as e:
        print(f"{Fore.YELLOW}Error providing downloads: {str(e)}")
        print(f"{Fore.YELLOW}Files are saved in the {OUTPUT_DIR} directory")

if __name__ == "__main__":
    main()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.6/150.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 36.4 MB/s eta 0:00:00

╔══════════════════════════════════════════════════════════════════╗
║                                                                  ║
║  SIMPLE BITCOIN KEYSPACE ANALYZER                            ║
║  Analyzing Range 40M-71M (No Batching)                      ║
╚══════════════════════════════════════════════════════════════════╝
    
Loading target Bitcoin addresses...
Upload a text file with Bitcoin addresses (one per line)


In [1]:
#!/usr/bin/env python3
# Simple Bitcoin Keyspace Analyzer - No Batches
# Tracks all 999 addresses in memory, supports compressed and uncompressed P2PKH keys

# Install required packages
# Note: This is commented out as it should be run in Colab cell separately
!pip install -q base58 colorama tqdm ecdsa pycryptodome

import hashlib
import base58
import os
import time
from colorama import Fore, init
from tqdm import tqdm
import binascii
import ecdsa
from Crypto.Hash import RIPEMD160

# Initialize colorama
init(autoreset=True)

# Configuration
RANGE_START = 40_000_000  # Start of key range to analyze
RANGE_END = 71_000_000    # End of key range to analyze
STEP_SIZE = 100_000       # Progress updates every this many keys
OUTPUT_DIR = "keyspace_analysis"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# File paths
LOG_FILE = f"{OUTPUT_DIR}/analysis_log.txt"
NEW_BESTS_FILE = f"{OUTPUT_DIR}/new_bests.csv"
FINAL_REPORT = f"{OUTPUT_DIR}/final_report.txt"

# Function to generate Bitcoin address from private key integer
def generate_address(private_key_int):
    try:
        # Format private key as hex
        private_key_hex = format(private_key_int, '064x')

        # Convert to ECDSA key
        sk = ecdsa.SigningKey.from_string(bytes.fromhex(private_key_hex), curve=ecdsa.SECP256k1)
        vk = sk.get_verifying_key()

        # Generate uncompressed public key (65 bytes: 0x04 + x + y)
        public_key_uncompressed = b'\x04' + vk.to_string()

        # Generate compressed public key (33 bytes: 0x02 or 0x03 + x)
        x_coord = vk.to_string()[:32]
        y_coord = vk.to_string()[32:]
        prefix = b'\x02' if y_coord[-1] % 2 == 0 else b'\x03'
        public_key_compressed = prefix + x_coord

        # Process both uncompressed and compressed keys
        addresses = []
        for public_key, key_type in [(public_key_uncompressed, "uncompressed"), (public_key_compressed, "compressed")]:
            # Create hash
            sha256_hash = hashlib.sha256(public_key).digest()

            # Apply RIPEMD160
            ripemd160_hasher = RIPEMD160.new()
            ripemd160_hasher.update(sha256_hash)
            hash160 = ripemd160_hasher.digest()

            # Create address with version and checksum
            versioned_hash = b'\x00' + hash160
            checksum = hashlib.sha256(hashlib.sha256(versioned_hash).digest()).digest()[:4]
            address_bytes = versioned_hash + checksum

            # Encode as Base58
            address = base58.b58encode(address_bytes).decode('utf-8')
            addresses.append((address, hash160, key_type))

        return addresses  # Returns [(address, hash160, key_type), ...]
    except Exception as e:
        print(f"Error generating address for key {private_key_int}: {str(e)}")
        return []

# Function to calculate Hamming distance between two binary objects
def hamming_distance(hash1, hash2):
    try:
        # Ensure both hashes are of same length
        if not hash1 or not hash2 or len(hash1) != len(hash2):
            return 160  # Maximum possible Hamming distance for 160-bit hash

        # Calculate Hamming distance bit by bit
        hamming = 0
        for a, b in zip(hash1, hash2):
            xor_byte = a ^ b
            # Count bits in XOR result
            while xor_byte:
                hamming += xor_byte & 1
                xor_byte >>= 1
        return hamming
    except Exception as e:
        print(f"Error calculating Hamming distance: {str(e)}")
        return 160

# Simple function to load Bitcoin addresses
def load_addresses():
    print(f"{Fore.CYAN}Loading target Bitcoin addresses...")

    try:
        # Check if addresses.txt exists
        if os.path.exists("addresses.txt"):
            with open("addresses.txt", 'r') as f:
                addresses = [line.strip() for line in f if line.strip()]
        else:
            # In Colab, we need to handle file upload carefully
            print(f"{Fore.CYAN}Please upload a text file with Bitcoin addresses (one per line)")
            from google.colab import files
            try:
                uploaded = files.upload()
                if not uploaded:
                    print(f"{Fore.RED}No file uploaded. Exiting.")
                    return [], {}

                filename = list(uploaded.keys())[0]
                # Save uploaded file as addresses.txt
                with open("addresses.txt", 'wb') as f:
                    f.write(uploaded[filename])
                with open("addresses.txt", 'r') as f:
                    addresses = [line.strip() for line in f if line.strip()]
            except Exception as e:
                print(f"{Fore.RED}Error uploading file: {str(e)}")
                return [], {}

        # Validate addresses and extract hash160 values
        valid_addresses = []
        hash160s = {}

        for addr in addresses:
            if not addr.startswith('1'):  # Only P2PKH addresses
                continue

            try:
                decoded = base58.b58decode(addr)
                if len(decoded) != 25:
                    continue

                hash160 = decoded[1:-4]  # Extract RIPEMD160 hash
                valid_addresses.append(addr)
                hash160s[addr] = hash160
            except Exception as e:
                print(f"Error decoding address {addr}: {str(e)}")
                continue

        if not valid_addresses:
            print(f"{Fore.RED}No valid P2PKH addresses found.")
            return [], {}

        print(f"{Fore.GREEN}Loaded {len(valid_addresses)} valid addresses for comparison.")
        return valid_addresses, hash160s

    except Exception as e:
        print(f"{Fore.RED}Error loading addresses: {str(e)}")
        return [], {}

# Function to initialize result tracking
def initialize_tracking(addresses):
    # Initialize tracking dictionaries - all start with max distance
    best_distances = {addr: 160 for addr in addresses}
    best_keys = {addr: None for addr in addresses}
    best_addresses = {addr: None for addr in addresses}
    best_key_types = {addr: None for addr in addresses}  # Track compressed/uncompressed

    # Initialize log file
    with open(LOG_FILE, 'w') as f:
        f.write("=== BITCOIN KEYSPACE ANALYSIS - SIMPLE VERSION ===\n")
        f.write(f"Started: {time.ctime()}\n")
        f.write(f"Analyzing key range: {RANGE_START:,} to {RANGE_END:,}\n")
        f.write(f"Target addresses: {len(addresses)}\n\n")
        f.write("Target addresses:\n")
        for addr in addresses:
            f.write(f"  {addr}\n")
        f.write("\n")

    # Initialize new bests file
    with open(NEW_BESTS_FILE, 'w') as f:
        f.write("Timestamp,TargetAddress,PrivateKey,GeneratedAddress,HammingDistance,PreviousBest,Improvement,Range,KeyType\n")

    return best_distances, best_keys, best_addresses, best_key_types

# Main analysis function
def analyze_range(start_key, end_key, target_addresses, target_hash160s):
    # Initialize tracking
    best_distances, best_keys, best_addresses, best_key_types = initialize_tracking(target_addresses)

    # Stats tracking
    improvements_count = 0
    start_time = time.time()

    # Create progress bar
    total_keys = end_key - start_key
    progress_bar = tqdm(total=total_keys, desc="Analyzing keys")

    # Process each key in range
    current_key = start_key
    last_update = start_time

    while current_key < end_key:
        # Generate addresses (both compressed and uncompressed) from private key
        address_results = generate_address(current_key)

        if address_results:
            # Compare against all target addresses
            for target_addr, target_hash160 in target_hash160s.items():
                for address, hash160, key_type in address_results:
                    # Calculate Hamming distance
                    dist = hamming_distance(hash160, target_hash160)

                    # Check if this is a new best
                    if dist < best_distances[target_addr]:
                        prev_dist = best_distances[target_addr]
                        improvement = prev_dist - dist

                        # Update best results
                        best_distances[target_addr] = dist
                        best_keys[target_addr] = current_key
                        best_addresses[target_addr] = address
                        best_key_types[target_addr] = key_type

                        # Get current range in millions
                        current_range = f"{current_key // 1_000_000}M"

                        # Print new best immediately
                        print(f"\n{Fore.GREEN}🔥 NEW BEST! Target: {target_addr}")
                        print(f"{Fore.GREEN}    Previous best: {prev_dist} bits")
                        print(f"{Fore.GREEN}    New best: {dist} bits (improved by {improvement} bits)")
                        print(f"{Fore.GREEN}    Key: {current_key} (hex: {format(current_key, '064x')})")
                        print(f"{Fore.GREEN}    Generated address: {address}")
                        print(f"{Fore.GREEN}    Key type: {key_type}")
                        print(f"{Fore.GREEN}    Range: {current_range}")

                        # Log the new best
                        timestamp = time.strftime("%Y-%m-%d %H:%M:%S")

                        with open(NEW_BESTS_FILE, 'a') as f:
                            f.write(f"{timestamp},{target_addr},{format(current_key, '064x')}," +
                                    f"{address},{dist},{prev_dist},{improvement},{current_range},{key_type}\n")

                        with open(LOG_FILE, 'a') as f:
                            f.write(f"[{timestamp}] NEW BEST for {target_addr}\n")
                            f.write(f"  Previous best: {prev_dist} bits\n")
                            f.write(f"  New best: {dist} bits (improved by {improvement} bits)\n")
                            f.write(f"  Key: {current_key} (hex: {format(current_key, '064x')})\n")
                            f.write(f"  Generated address: {address}\n")
                            f.write(f"  Key type: {key_type}\n")
                            f.write(f"  Range: {current_range}\n\n")

                        improvements_count += 1

        # Update progress every STEP_SIZE keys
        if current_key % STEP_SIZE == 0:
            # Update progress bar
            progress_bar.update(STEP_SIZE)

            # Print stats every minute
            current_time = time.time()
            if current_time - last_update >= 60:
                keys_processed = current_key - start_key
                elapsed = current_time - start_time
                keys_per_second = keys_processed / elapsed if elapsed > 0 else 0
                under_40_count = sum(1 for dist in best_distances.values() if dist < 40)

                range_str = f"{(current_key // 1_000_000)}M"
                print(f"\n{Fore.CYAN}Progress Update - Range: {range_str}")
                print(f"{Fore.CYAN}  Keys processed: {keys_processed:,} ({keys_per_second:.2f} keys/sec)")
                print(f"{Fore.CYAN}  Improvements found: {improvements_count}")
                print(f"{Fore.CYAN}  Addresses with Hamming < 40: {under_40_count}/{len(target_addresses)}")

                # Estimate remaining time
                remaining_keys = end_key - current_key
                est_remaining_time = remaining_keys / keys_per_second if keys_per_second > 0 else 0
                print(f"{Fore.CYAN}  Estimated remaining time: {est_remaining_time:.2f} seconds")

                last_update = current_time

        # Move to next key
        current_key += 1

    # Close progress bar
    progress_bar.close()

    # Generate final report
    generate_report(target_addresses, best_distances, best_keys, best_addresses, best_key_types)

    # Show final stats
    elapsed = time.time() - start_time
    print(f"\n{Fore.GREEN}Analysis completed in {elapsed:.2f} seconds")
    print(f"{Fore.GREEN}Found {improvements_count} improvements")
    under_40_count = sum(1 for dist in best_distances.values() if dist < 40)
    print(f"{Fore.GREEN}Addresses with Hamming < 40: {under_40_count}/{len(target_addresses)}")

    # Return results
    return best_distances, best_keys, best_addresses, best_key_types

# Function to generate final report
def generate_report(target_addresses, best_distances, best_keys, best_addresses, best_key_types):
    try:
        with open(FINAL_REPORT, 'w') as f:
            f.write("=== BITCOIN KEYSPACE ANALYSIS - FINAL REPORT ===\n")
            f.write(f"Generated: {time.ctime()}\n")
            f.write(f"Key range analyzed: {RANGE_START:,} to {RANGE_END:,}\n")
            f.write(f"Target addresses: {len(target_addresses)}\n\n")

            # Overall statistics
            min_hamming = min(best_distances.values()) if best_distances else 160
            under_40_count = sum(1 for dist in best_distances.values() if dist < 40)

            f.write(f"Overall minimum Hamming distance: {min_hamming} bits\n")
            f.write(f"Addresses with Hamming distance < 40: {under_40_count}/{len(target_addresses)}\n\n")

            # Sort addresses by Hamming distance
            sorted_addresses = sorted(
                [(addr, best_distances[addr]) for addr in target_addresses],
                key=lambda x: x[1]
            )

            # Best results section
            f.write("Best results:\n\n")

            for addr, dist in sorted_addresses:
                if dist < 160:  # Only include addresses with actual matches
                    f.write(f"Target address: {addr}\n")
                    f.write(f"Best Hamming distance: {dist} bits\n")
                    if best_keys[addr] is not None:
                        f.write(f"Private key: {best_keys[addr]} (hex: {format(best_keys[addr], '064x')})\n")
                        f.write(f"Generated address: {best_addresses[addr]}\n")
                        f.write(f"Key type: {best_key_types[addr]}\n")
                    f.write("\n")

            # Summary by Hamming distance ranges
            f.write("Summary by Hamming distance ranges:\n\n")
            ranges = [(0, 30), (30, 35), (35, 40), (40, 45), (45, 50), (50, 160)]

            for low, high in ranges:
                count = sum(1 for dist in best_distances.values() if low <= dist < high)
                f.write(f"  {low}-{high} bits: {count} addresses\n")

        print(f"{Fore.GREEN}Final report saved to {FINAL_REPORT}")

    except Exception as e:
        print(f"{Fore.RED}Error generating report: {str(e)}")

# Main function
def main():
    print(f"""
{Fore.CYAN}╔══════════════════════════════════════════════════════════════════╗
{Fore.CYAN}║                                                                  ║
{Fore.CYAN}║  {Fore.GREEN}SIMPLE BITCOIN KEYSPACE ANALYZER{Fore.CYAN}                            ║
{Fore.CYAN}║  {Fore.YELLOW}Analyzing Range 40M-71M (No Batching, Compressed/Uncompressed){Fore.CYAN}  ║
{Fore.CYAN}╚══════════════════════════════════════════════════════════════════╝
    """)

    # Load addresses
    target_addresses, target_hash160s = load_addresses()

    if not target_addresses:
        print(f"{Fore.RED}No valid addresses loaded. Exiting.")
        return

    print(f"{Fore.GREEN}Starting simple keyspace analysis - showing only new best matches")
    print(f"{Fore.GREEN}Range: {RANGE_START:,} to {RANGE_END:,}")
    print(f"{Fore.GREEN}Tracking {len(target_addresses)} addresses in memory")

    # Run analysis
    best_distances, best_keys, best_addresses, best_key_types = analyze_range(
        RANGE_START, RANGE_END, target_addresses, target_hash160s
    )

    # Provide download links
    print(f"\n{Fore.CYAN}Download results:")
    try:
        from google.colab import files
        files.download(FINAL_REPORT)
        files.download(NEW_BESTS_FILE)
        files.download(LOG_FILE)
    except Exception as e:
        print(f"{Fore.YELLOW}Error providing downloads: {str(e)}")
        print(f"{Fore.YELLOW}Files are saved in the {OUTPUT_DIR} directory")

if __name__ == "__main__":
    main()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.6/150.6 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 72.8 MB/s eta 0:00:00

╔══════════════════════════════════════════════════════════════════╗
║                                                                  ║
║  SIMPLE BITCOIN KEYSPACE ANALYZER                            ║
║  Analyzing Range 40M-71M (No Batching, Compressed/Uncompressed)  ║
╚══════════════════════════════════════════════════════════════════╝
    
Loading target Bitcoin addresses...
Please upload a text file with Bitcoin addresses (one per line)


KeyboardInterrupt: 